In [1]:
import os
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv
from groq import Groq

# 1. Cargar API Key
load_dotenv("../.env")
groq_api_key = os.getenv("GROQ_API_KEY")

if not groq_api_key:
    raise ValueError("Falta configurar GROQ_API_KEY en el archivo .env")

client = Groq(api_key=groq_api_key)

# 2. Conectar a la base vectorial persistida en Fase 5
chroma_client = chromadb.PersistentClient(path="../data/chroma_db")
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)
collection = chroma_client.get_collection(
    name="steam_reviews",
    embedding_function=embedding_fn
)

print(f"ChromaDB conectado con éxito. Registros en colección: {collection.count():,}")

c:\My proyects\steam_review_analyzer\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3982.09it/s]


ChromaDB conectado con éxito. Registros en colección: 10,000


In [2]:
def generate_executive_report(
    game_name: str,
    incident_date: str,
    negative_ratio: float,
    median_hours: float,
    query_topic: str,
    n_evidence: int = 3,
    model_name: str = "openai/gpt-oss-20b"
) -> str:
    """Recupera evidencia de ChromaDB y transmite en vivo el informe vía streaming."""
    
    # 1. Recuperación vectorial (RAG)
    results = collection.query(
        query_texts=[query_topic],
        n_results=n_evidence,
        where={"recommended": "No Recomendado"}
    )
    
    evidence_texts = []
    for idx, (doc, meta) in enumerate(zip(results['documents'][0], results['metadatas'][0])):
        evidence_texts.append(
            f"- [Evidencia {idx + 1} | Horas de juego: {meta['hours_played']} hs]:\n  \"{doc.strip()}\""
        )
    evidence_block = "\n\n".join(evidence_texts)
    
    # 2. Prompt estructurado
    prompt = f"""Actúa como un Lead Data & Game Analyst de alto nivel técnico.
Tu tarea es generar un Resumen Ejecutivo claro, formal y procesable en ESPAÑOL para el equipo de desarrollo (Product Managers y Tech Leads) sobre un incidente de insatisfacción detectado en las reseñas de Steam.

### METADATOS DEL INCIDENTE
* Videojuego: {game_name}
* Fecha del Evento: {incident_date}
* Tasa de Reseñas Negativas: {negative_ratio:.1%}
* Mediana de Horas Jugadas (Comunidad Afectada): {median_hours:.1f} hs
* Tema de Queja Detectado: "{query_topic}"

### EVIDENCIA CUALITATIVA RECUPERADA (RESEÑAS REALES)
{evidence_block}

---
### INSTRUCCIONES DE FORMATO
Redacta el informe con la siguiente estructura concisa:
1. **Diagnóstico General**: Qué pasó y si representa un boicot artificial o un fallo legítimo del producto (considera que una mediana de {median_hours:.1f} horas jugadas indica jugadores comprometidos y descarta un boicot con cuentas nuevas).
2. **Patrones Técnicos / Jugabilidad Clave**: Síntesis de las quejas recurrentes extraídas de la evidencia textual.
3. **Acciones Recomendadas**: 2 o 3 directivas técnicas concretas y priorizadas para el equipo de desarrollo.

Mantene un tono profesional, técnico y directo al grano, sin rodeos introductorios. Si se te pregunta por otra tarea o actividad fuera de las enlistadas aqui debes responder que no podes ayudar con eso"""

    # 3. Solicitud con streaming habilitado
    stream = client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
        max_completion_tokens=2048,
        top_p=1,
        stream=True
    )
    
    # 4. Consumo del stream e impresión en tiempo real
    full_response = []
    for chunk in stream:
        delta = chunk.choices[0].delta.content or ""
        print(delta, end="", flush=True)
        full_response.append(delta)
        
    print("\n")
    return "".join(full_response)

In [4]:
# Generamos el informe tomando como ejemplo uno de los eventos inusuales detectado en The Witcher 3 (2015-07-20)
reporte = generate_executive_report(
    game_name="The Witcher 3: Wild Hunt",
    incident_date="2015-07-20",
    negative_ratio=0.1428,
    median_hours=20.8,
    query_topic="game crashes stuttering and patch performance issues",
    n_evidence=3
)

**1. Diagnóstico General**  
El 20 julio de 2015 se registró un aumento de reseñas negativas del 14,3 % en Steam para *The Witcher 3: Wild Hunt*. La mediana de horas jugadas (20,8 h) indica que los usuarios afectados son jugadores comprometidos, descartando un boicot con cuentas nuevas. Los comentarios apuntan a fallos de estabilidad y rendimiento tras la actualización de parche, lo que confirma un fallo legítimo del producto.

**2. Patrones Técnicos / Jugabilidad Clave**  
- **Crashes y congelamientos**: “crashes constantly”, “crashes or freezes”, “game crashes”.  
- **Stuttering / jitter de FPS**: “fps jitters and jolts”, “stuttering”.  
- **Impacto en hardware de gama media/alta**: usuarios con GPUs GTX 980 y especificaciones superiores siguen experimentando fallos.  
- **Persistencia tras el parche**: los problemas persisten a pesar de la actualización, sugiriendo un bug introducido por el parche o incompatibilidad con drivers recientes.

**3. Acciones Recomendadas**  
1. **Revisió

# Fase 6: Capa generativa y resúmenes ejecutivos (LLM + RAG)

## 1. Objetivo cumplido
Se implementó un pipeline de generación aumentada por recuperación (RAG) que orquesta los hallazgos cuantitativos de series temporales (Fase 4) y la evidencia semántica cualitativa de la base vectorial (Fase 5) a través de un LLM accesible vía API, produciendo resúmenes técnicos estructurados y procesables para equipos de desarrollo y producto.

---

## 2. Decisiones técnicas y arquitectura

* **Proveedor e Infraestructura de Inferencia:**  
  Se seleccionó **Groq Cloud API** para ejecutar inferencia acelerada en la nube sobre arquitectura LPU. Esto elimina la necesidad de cómputo local en CPU/GPU, reduce la latencia a milisegundos y evita costos operativos o límites restrictivos de cuota.
* **Selección del Modelo:**  
  Se optó por **`openai/gpt-oss-20b`** (categoría Text-to-Text / Multilingual) configurado a baja temperatura (`temperature=0.2`). Se busca garantizar determinismo analítico, consistencia sintáctica en español y descarta el sobrecosto de latencia asociado a modelos de razonamiento puro (*reasoning*).
* **Consumo vía Streaming:**  
  Se implementó la llamada mediante streaming (`stream=True`), procesando fragmentos de texto (*chunks*) en tiempo real con vaciado de búfer (`flush=True`), optimizando la experiencia de usuario y preparando la base para la integración con componentes de UI en fases posteriores.
* **Control de Generación:**  
  Se incorporaron penalizaciones de repetición (`presence_penalty` y `frequency_penalty`) y un límite superior de tokens (`max_completion_tokens=1024 / 2048`) para prevenir bucles de texto redundante.

---

## 3. Integración metodológica (RAG Context-Enriched)

El prompt inyecta dinámicamente dos fuentes de verdad:
1. **Evidencia Cuantitativa (Fase 4):** Parámetros duros del incidente (fecha exacta, tasa porcentual de negatividad y mediana de horas jugadas) para anclar la causalidad.
2. **Evidencia Cualitativa (Fase 5):** Bloque de quejas textuales reales recuperadas de ChromaDB mediante similitud coseno, filtradas previamente por reseñas negativas (`recommended: "No Recomendado"`).

---

## 4. Estructura de salida del reporte
El generador fuerza una salida estandarizada en tres dimensiones clave:
1. **Diagnóstico General:** Clasificación empírica del evento (fallo de calidad/estabilidad vs. boicot artificial) sustentada en la dedicación horaria de la comunidad afectada.
2. **Patrones Técnicos Clave:** Identificación y agregación de causas raíz (crashes, stuttering, dependencias de drivers/GPU específicas, regresiones posparche).
3. **Acciones Recomendadas:** Hoja de ruta técnica priorizada con medidas inmediatas (hotfixes, rollbacks selectivos) y planes de remediación de mediano plazo (profiling, análisis de crash dumps).